In [2]:
import pandas as pd
import numpy as np

In [3]:
#loading tables
users=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name="Users")
activity=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name=1)
transactions=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name=2)
support=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name=3)
marketing=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name=4)
subscription=pd.read_excel("User_Retention_Portfolio_Dataset.xlsx",sheet_name=5)

In [5]:
tables = {
    "users": users,
    "activity": activity,
    "transactions": transactions,
    "support": support,
    "marketing": marketing,
    "subscription": subscription
}

for name, df in tables.items():
    print("\n", "="*50)
    print(name)
    print("Shape:", df.shape)
    print("Duplicates:", df.duplicated().sum())
    print("Null values:")
    print(df.isnull().sum())


users
Shape: (5000, 12)
Duplicates: 0
Null values:
user_id                   0
signup_date               0
country                   0
state                     0
age                       0
gender                    0
device                    0
acquisition_channel       0
subscription_type         0
churned                   0
churn_date             3394
age_group                 0
dtype: int64

activity
Shape: (150000, 8)
Duplicates: 0
Null values:
activity_id        0
user_id            0
activity_date      0
session_minutes    0
pages_viewed       0
login_count        0
feature_used       0
device             0
dtype: int64

transactions
Shape: (50000, 8)
Duplicates: 0
Null values:
transaction_id      0
user_id             0
transaction_date    0
amount              0
payment_method      0
status              0
coupon_code         0
refund              0
dtype: int64

support
Shape: (18000, 8)
Duplicates: 0
Null values:
ticket_id           0
user_id             0
created_date    

In [46]:
for i, j in tables.items():
    print(i, "->", j.columns[0])

users -> user_id
activity -> activity_id
transactions -> transaction_id
support -> ticket_id
marketing -> campaign_id
subscription -> history_id


In [6]:
print("Primary_Keys","\n")
for i,j in tables.items():
    print(i,":",j.iloc[:,0].nunique())

Primary_Keys 

users : 5000
activity : 150000
transactions : 50000
support : 18000
marketing : 5000
subscription : 20000


In [7]:
#checking relationship
print(activity["user_id"].isin(users["user_id"]).mean())
print(transactions["user_id"].isin(users["user_id"]).mean())
print(support["user_id"].isin(users["user_id"]).mean())
print(subscription["user_id"].isin(users["user_id"]).mean())

1.0
1.0
1.0
1.0


In [8]:
# date range
print("User signup:")
print(users["signup_date"].min(), users["signup_date"].max(),"\n")

print("Activity:")
print(activity["activity_date"].min(), activity["activity_date"].max(),"\n")

print("Transactions:")
print(transactions["transaction_date"].min(), transactions["transaction_date"].max(),"\n")

User signup:
2024-01-01 00:00:00 2024-12-31 00:00:00 

Activity:
2024-01-01 00:00:00 2025-06-29 00:00:00 

Transactions:
2024-01-02 00:00:00 2025-06-29 00:00:00 



In [9]:

users["signup_month"]=users["signup_date"].dt.to_period("M")

users[["user_id","signup_date","signup_month"]].head()

,user_id,signup_date,signup_month
0,1,2024-11-23,2024-11
1,2,2024-08-04,2024-08
2,3,2024-11-28,2024-11
3,4,2024-05-22,2024-05
4,5,2024-06-25,2024-06


In [11]:
#activiry_cohort
activity_cohort=activity.merge(users[["user_id","signup_month"]],on="user_id",how="inner")
activity_cohort.head()

,activity_id,user_id,activity_date,session_minutes,pages_viewed,login_count,feature_used,device,signup_month
0,1,826,2024-02-18,10,13,3,Browse,Android,2024-01
1,5620,826,2024-06-17,14,10,1,Profile,Android,2024-01
2,8418,826,2024-05-02,22,20,3,Watch,Android,2024-01
3,10032,826,2024-05-18,10,23,3,Watch,Android,2024-01
4,10585,826,2024-07-08,6,4,4,Profile,Android,2024-01


In [12]:
activity_cohort["activity_month"]=activity_cohort["activity_date"].dt.to_period("M")
activity_cohort.head()

,activity_id,user_id,activity_date,session_minutes,pages_viewed,login_count,feature_used,device,signup_month,activity_month
0,1,826,2024-02-18,10,13,3,Browse,Android,2024-01,2024-02
1,5620,826,2024-06-17,14,10,1,Profile,Android,2024-01,2024-06
2,8418,826,2024-05-02,22,20,3,Watch,Android,2024-01,2024-05
3,10032,826,2024-05-18,10,23,3,Watch,Android,2024-01,2024-05
4,10585,826,2024-07-08,6,4,4,Profile,Android,2024-01,2024-07


In [13]:
activity_cohort["cohort_month"]=((activity_cohort["activity_month"].dt.year - activity_cohort["signup_month"].dt.year)*12 + (activity_cohort["activity_month"].dt.month - activity_cohort["signup_month"].dt.month))
activity_cohort.head()

,activity_id,user_id,activity_date,session_minutes,pages_viewed,login_count,feature_used,device,signup_month,activity_month,cohort_month
0,1,826,2024-02-18,10,13,3,Browse,Android,2024-01,2024-02,1
1,5620,826,2024-06-17,14,10,1,Profile,Android,2024-01,2024-06,5
2,8418,826,2024-05-02,22,20,3,Watch,Android,2024-01,2024-05,4
3,10032,826,2024-05-18,10,23,3,Watch,Android,2024-01,2024-05,4
4,10585,826,2024-07-08,6,4,4,Profile,Android,2024-01,2024-07,6


In [14]:
#unique active users
cohort_data=activity_cohort.groupby(["signup_month","cohort_month"])["user_id"].nunique().reset_index().rename(columns={"user_id":"active_users"})
cohort_data

,signup_month,cohort_month,active_users
0,2024-01,0,351
1,2024-01,1,419
2,2024-01,2,420
3,2024-01,3,421
4,2024-01,4,422
...,...,...,...
79,2024-12,2,426
80,2024-12,3,427
81,2024-12,4,427
82,2024-12,5,427
